## Chapter 02 

## RADONIKA LIBRARY

We start building the library now. We will have the standard structure. We will come back to this later once the library is prepared.

## 2.1 PROCESS LAYER

---

The file `process.py` is a 2-D stochastic process engine. Instead of reading it from line 1 to the end, we will build it bottom-up. Starting with a concrete process, finding what logic repeats, extracting that logic into base classes and helpers, and gradually expanding into more complex math. 

Every object in `process.py` models a 2-D trajectory $(X_t, Y_t)$. Rather than writing separate simulation, density, and plotting code for every single random process, we will make a file that has a separate definition for things that we will use again and again.

```text
Concrete Processes (normal, bm, poisson, stocks...)
├── Defined using their unique maths, how to step forward
│
└── Intermediate Wrappers (_IID)
    ├── Groups processes that are iid
    │
    └── Class (Process)
        ├── Provides the universal engine: paths(), sample(), pdf(), measure tilting, & UI plots
        │
        └── 4 tiny helpers (_pair, _vol, _bvn_pdf, _gauss_paths)
```

## 2.1.1 Two-Dimensional iid Processes

Since we plan on making $iid$ Cauchy, Normal, Uniform etc, the main underlying engine of generating the array etc can be created separately under something along the lines of a engine that draws randomly to make a $iid$ process. We can name it `iid` ofcourse. Hence, they will all form a subclass as `class normal(iid)`, `class cauchy(iid)`, etc. Then the `class iid` can have the mechanisms to generate the process and look at distributions at several time points etc. This stucture is fine but there are more technicalities. In this library, we are going to have processes that are non-$iid$s too. We will want to simulate these paths too and also look at the distributions at several time points. What now? 

There is an easy fix for this. Make `iid` itself a subclass of an even more fundamental class named `Process`. This will have all forms of the functions we want to do to our processes. It will have methods such as `simulate` that will be called again and again and defined as per our need. This flexibility is needed since the way to simulate $iid$ processes is obviously different from the way to simulate Brownian motion.

In [1]:
class normal:
    def __init__(self, mean=0.0, sigma=1.0, seed=None):
        m, s = _pair(mean, "mean"), _pair(sigma, "sigma")                  #the _pair helper is used
        super().__init__([stats.norm(m[k], s[k]) for k in range(2)], seed,
                         f"normal(mean={fmt(m)}, sigma={fmt(s)})")

Due to some technicalities, we will use `_IID` instead of `iid` (we will see why later). More precisely, above code should then be `normal(_IID):` where `_IID` will simulate the trajectories. That will further need `Process` as we said before. So, let us start there now, from making the `class Process`.

```text
shape = (n, len(times), 2)
         │      │        └─ the two coordinates, X and Y
         │      └────────── how many time points
         └───────────────── how many paths
```
If we had to do this with nothing else, the whole Process would be this:

In [2]:
class Process:
    def simulate(self, n, times, start, rng):
        raise NotImplementedError

`Process` itself has no idea how to simulate anything. It says “every process must have a `simulate`” and raises `NotImplementedError` if a subclass forgets. Each specific process (Cauchy, Brownian motion, and so on) fills in the actual recipe. We will now make all these small building blocks for the `class Process` such as `_rng`, `simulate`, `_start`, `paths` etc. We have already discussed what `rng` does in Python. Let us see what the others are doing one by one.

One rule is written in the docstring that `times[0]` must be 0, and the first column of the output must equal `start`. Every path starts at the starting point. The `simulate` also needs `start`, the point `(x0, y0)` where every path begins. The user might give it in a few different ways, or not at all, so the process needs one place that turns whatever they gave into a clean array.

In [3]:
def _start(self, start):
    s = self.default_start if start is None else start
    s = np.asarray(s, float)
    if s.ndim == 0:
        s = np.repeat(s, self.dim)
    if s.shape != (self.dim,):
        raise ValueError(f"start must have {self.dim} coordinates")
    return s

It does four small jobs, in order:

1. Fill in a default. If the user passed nothing (None), use `self.default_start`, which is (0.0, 0.0)` for most processes. Stock prices override it to `(1.0, 1.0)`, because a price starting at 0 would be stuck at 0 forever.
2. Convert to a NumPy array of floats. A tuple `(0, 0)` becomes array `([0., 0.])`.
3. Allow a shortcut. If the user passed a single number like 5, `s.ndim == 0` is true, and `np.repeat(s, 2)` turns it into `[5., 5.]`. So `start=5` means “start both coordinates at 5”.
4. Check the shape. After all that, the result must be exactly length 2. If not, it raises a clear error instead of failing mysteriously later.

We now have `_rng` and `_start`. We also have `simulate`. But calling `simulate` directly is clumsy because the user would have to build times, start and rng by hand.

`times = np.linspace(0, 1, 201)`

`X = process.simulate(20, times, process._start(None), process._rng(None))`

So `Process` gives a friendly method that does all of that named `paths`.

In [4]:
def paths(self, n=20, T=1.0, steps=200, start=None, seed=None):
    times = np.linspace(0.0, T, steps + 1)
    return times, self.simulate(n, times, self._start(start), self._rng(seed))

Note that `paths` is using `simulate` that further has `_rng` and `_start` that we built before. Every argument has a default, so `process.paths()` alone already works giving 20 paths, on $[0, 1]$, 200 steps, starting at the origin. So far we have:  

In [5]:
import numpy as np
from scipy import stats

class Process:
    """Base class: every process knows how to simulate itself and (where possible) its law."""

    name = "process"
    dim = 2
    default_start = (0.0, 0.0)

    def __init__(self, seed=None):
        self.seed = seed

    def __repr__(self):
        return self._desc

    def _start(self, start):
        """Validates or assigns default starting vector (0.0, 0.0)."""
        s = self.default_start if start is None else start
        s = np.asarray(s, float)
        if s.ndim == 0:
            s = np.repeat(s, self.dim)
        if s.shape != (self.dim,):
            raise ValueError(f"start must have {self.dim} coordinates")
        return s

    def _rng(self, seed=None):
        return np.random.default_rng(self.seed if seed is None else seed)

    def simulate(self, n, times, start, rng):
        """Paths of shape (n, len(times), dim); times[0] must be 0 and X_0 = start."""
        raise NotImplementedError

    def paths(self, n=20, T=1.0, steps=200, start=None, seed=None):
        """Simulate n paths on [0, T]; returns (times, X) with X of shape (n, steps+1, 2)."""
        times = np.linspace(0.0, T, steps + 1)
        return times, self.simulate(n, times, self._start(start), self._rng(seed))    

`paths()` gives you whole trajectories. Often you want a different question: “at time t = 3, what values does X_t take?” To see that distribution, you need many independent draws of (X_3, Y_3), with no need for the path leading up to it. This is the law of the process at that time (similar to the 2D case). The simplest way is to simulate paths up to time t and keep only the last time point. For that we need `sample` and `sample_slices`. The cloud of points is a picture of the law. It’s dense where the process likely lands and thin where it rarely does and that is what `sample` will do. 

Where does `sample_slices` come into the picture? We will make it like an animation that shows the law evolving over time. For that we need a process to simulate the cloud at several time points and that is what `sample_slice` will do. `sample_slice` can create many clouds over many time points and `sample` just looks at them one by one.

In [6]:
def sample_slices(self, times, n, rng, start):
    X = self.simulate(n, np.r_[0.0, np.asarray(times, float)], start, rng)
    return np.transpose(X[:, 1:, :], (1, 0, 2))

Recall, $X[3, 10, 1]$ means “path number 3, at the 10th time point, the Y coordinate”. So, `X[:, 1:, :]` means from the time array we are dropping the initial slice, start that array from 1. This is because we had `np.r_` that added $0$ in begining of `times`. So, something like $[1, 2, 3]$ became $[0, 1, 2, 3]$ but now we have to remove the initial time slice. The $0$ was added since that is how simulate works and the form of inputs are predefined. Once that simulation is done, we do need to return those time slices only that the user asked. The transpose here exchanges the first two component of $X$[path number, times, coordiate number] to $X$[times, path number, coordiate number]. This will read "(slice, n, 2)" which is used in next part.

In [7]:
def sample_slices(self, times, n, rng, start):
    X = self.simulate(n, np.r_[0.0, np.asarray(times, float)], start, rng)
    return np.transpose(X[:, 1:, :], (1, 0, 2))
    
def sample(self, t, n=10000, start=None, seed=None):
    return self.sample_slices([t], n, self._rng(seed), self._start(start))[0]

The `[0]` at the end of the def of `sample` is simply because sample took only one time as entry, namely, `[t]` and then we want to look at them one by one. So, `[0]` says move along the $0^{th}$ coordinate (time) one by one; so basically instead of stacks of clouds, it is just one cloud at a time. Now we have the updated class as:

In [8]:
class Process:
    """Base class: every process knows how to simulate itself and (where possible) its law."""

    name = "process"
    dim = 2
    default_start = (0.0, 0.0)

    def __init__(self, seed=None):
        self.seed = seed

    def __repr__(self):
        return self._desc

    def _start(self, start):
        """Validates or assigns default starting vector (0.0, 0.0)."""
        s = self.default_start if start is None else start
        s = np.asarray(s, float)
        if s.ndim == 0:
            s = np.repeat(s, self.dim)
        if s.shape != (self.dim,):
            raise ValueError(f"start must have {self.dim} coordinates")
        return s

    def _rng(self, seed=None):
        return np.random.default_rng(self.seed if seed is None else seed)

   #----------------------------------------------------------------------------- to be provided by subclasses
    def simulate(self, n, times, start, rng):
        """Paths of shape (n, len(times), dim); times[0] must be 0 and X_0 = start."""
        raise NotImplementedError

    #this is new and we will see what it does later
    def marginal(self, t, start, k):
        """Frozen scipy distribution of coordinate k at time t (when known in closed form)."""
        raise NotImplementedError
        
    #---------------------------------------------------------------------------------------------------------

    def paths(self, n=20, T=1.0, steps=200, start=None, seed=None):
        """Simulate n paths on [0, T]; returns (times, X) with X of shape (n, steps+1, 2)."""
        times = np.linspace(0.0, T, steps + 1)
        return times, self.simulate(n, times, self._start(start), self._rng(seed))   

    def sample_slices(self, times, n, rng, start):
        X = self.simulate(n, np.r_[0.0, np.asarray(times, float)], start, rng)
        return np.transpose(X[:, 1:, :], (1, 0, 2))
    
    def sample(self, t, n=10000, start=None, seed=None):
        return self.sample_slices([t], n, self._rng(seed), self._start(start))[0]

There will be other things to add here, we will come to them in due time. Notice we kept `simulate` and `marginal` as empty promises for now. They will be defined properly in later subclasses. In fact, we will make another class now named `_IID` that will define `simulate` and `marginal` properly. 

If we are not defining them yet why do we even put it here inside `Process` to begin with? This is because there are other methods inside the class that are defined on top of it. For example, `sample` used `simulate` even though exact way to do the simulation is yet given (we will see them under subclass `_IID`). Similarly, there are other methods such as `pdf` and `center` (we did not define them yet but they are going to be a part of `Process`) that will need `marginal` even though `marginal` is going to be defined propery inside subclass `_IID`.

Also, keeping them declared here as empty promises are helpful since we can keep calling them back and define them in different ways in various subclasses. For example, in `_IID` the exact mechanism behind `simulate` is going to be different from the mechanism we see in `simulate` while making Brownian-motion (that is not $iid$ ofcourse). So, the point is, we will see several ways to define `marginal` and `simulate` across various sub-classes but operations we want to do with these simulated processes (like `sample`, `sample_slices`) and the marginals can still be done since they are defined in an unified manner under the same roof `Process`.

Say we want to create $iid$ Cauchy. Using the built process will give us all those properties to out cauchy process. So, doing it by hand using only `Process` looks like:

In [9]:
class cauchy(Process):
    def simulate(self, n, times, start, rng):
        out = np.empty((n, len(times), 2))
        out[:, 0, :] = start                                    # time 0 = start point
        out[:, 1:, :] = rng.standard_cauchy((n, len(times)-1, 2))   # fresh noise afterwards
        return out

This works. Now if we want for Uniform we redo this:

In [10]:
class cauchy(Process):
    def simulate(self, n, times, start, rng):
        out = np.empty((n, len(times), 2))
        out[:, 0, :] = start                                    # time 0 = start point
        out[:, 1:, :] = rng.uniform((n, len(times)-1, 2))   # fresh noise afterwards
        return out

Now we can do this for all processes we want such as exponential, Laplace, Student-t, lognormal, etc. This is a signal to make a shared engine for $iid$ processes. So we make another `class` named `_IID(Process)` and it has `Process` as argument since we do not want to miss out on the properties we built so far. This is also the reason why we call this the subclass.

In [11]:
class _IID(Process):
    """X_t independent across time; coordinates independent scipy distributions."""

    def __init__(self, frozen, seed=None, desc="iid"):
        super().__init__(seed)
        self.frozen = list(frozen)
        self._desc = desc

    def simulate(self, n, times, start, rng):
        out = np.empty((n, len(times), 2))
        out[:, 0, :] = start
        for k in range(2):
            out[:, 1:, k] = self.frozen[k].rvs(size=(n, len(times) - 1), random_state=rng)
        return out

    def marginal(self, t, start, k):
        return self.frozen[k]

    def logpdf(self, X):
        return sum(self.frozen[k].logpdf(X[..., k]) for k in range(2))

As we said `simulate` will be defined again and again. Under `class _IID(Process)`, defining it is pretty easy; it is exactly same as the chunk of codes we were repeating again and again for Cauchy, uniform etc. 
1. `self.frozen[k].rvs(...)` replaces the hard-coded `rng.standard_cauchy`. It asks whatever distribution is stored for random draws, which is the whole point.
2. A loop over `k` in `range(2)`, because `self.frozen` is a list of two distributions, one for X (k=0) and one for Y (k=1). That lets X and Y have different parameters, for example X ~ Normal(0, 1) and Y ~ Normal(5, 2). Each fills its own slice `out[:, 1:, k]`. This gives us another degree of freedom hence.

As always `__init__` keeps in mind things about "self". For the base `class Process`, `__init__` was simply one ine long since it only needed `seed`. Here, for `class _IID(Process)` we will call back the seed from `Process` and add more details. `super().__init__(seed)` instaed of `self.seed = seed` because we are using the "parent's" `__init__` from `Process`. Lastly, `desc` is nothing mathematical, it is just a text label. If we type `print(p)`, Python shows the object’s `__repr__`, which returns `self._desc`. So with desc="cauchy", printing shows cauchy. This has no further deeper meaning.

Recall now, `marginal` was also declared as an empty promise inside `Process` like `simulate`. For $iid$ set up, distribution at any time $t$ is identical; so it is same as the stored distribtuion `self.frozen`. But keep track of the fact that it is not always going to be this easy. For example soon we will see the class for Brownian motion. There neither the `simulate` nor `marginal` will be this easy. In fact, this is the reason why we did the 1D processes first. Recall the marginal distribution of Brownian motion over time - the variance kept on increasing as we moved forward in time. So, here `marginal` at time $t$ will not be the stored distribtuion but a function of time $t$ itself. 

Lastly, the `logpdf` codes. This returns the log of the density at the points X. Because the X and Y coordinates are independent, the joint density is a product, $f(x, y) = f(x) · f(y)$. Taking logs turns the product into a sum, so we add the two log-densities. `X[..., k]` means “the k-th coordinate, whatever the other axes look like.”

Next we use `_IID` directly and see if it is still an inconvenience. For a Cauchy process we still have to write :

In [12]:
p = _IID([stats.cauchy(0, 1), stats.cauchy(0, 1)], seed=1, desc="cauchy")

`_IID` only ensures every $iid$ process now is one line long but we can make it even better. This was expected, I guess; the leading underscore I used in `_IID` is a hint that it is an internal engine, not something the user should touch. Now we give the user a cleaner front door to use named `iid`. Its job is to accept any scipy distribution and build the `_IID` for us. 

In [13]:
def iid(dist, seed=None):
    pair = list(dist) if isinstance(dist, (list, tuple)) else [dist, dist]
    return _IID(pair, seed, desc=f"iid({pair[0].dist.name})")

p = iid(stats.gamma(2), seed=1)
#notice how short it is now as compared to "p = _IID([stats.cauchy(0, 1), stats.cauchy(0, 1)], seed=1, desc="cauchy")"

`isinstance(input, (A, B))` is a standard code again. It asks if `input` is one of the two options `A` and `B`. If `input` is `A` or `B` it returns `True`. Above code says "If input is list or tuple, use them. Else repeat whatever we have twice."

What it still can’t do? `iid()` needs you to hand it a ready-made scipy distributions. For Cauchy we will write `iid(stats.cauchy(0, 1))`, which means knowing scipy’s parameter names.

In [14]:
p = iid([stats.norm(0, 1), stats.norm(5, 2)])    # a list, isinstace=True
p = iid((stats.norm(0, 1), stats.norm(5, 2)))    # a tuple works too, isinstance=True
p = iid(stats.gamma(1), seed=1)                  #not a tuple or a list, ininstance=False. So, it will be repeated twice

Now finally we have created all that we need for these $iid$ processes. We will generate $iid$ normal, $iid$ cauchy, $iid$ exponential, $iid$ uniform, $iid$ uniform, $iid$ student-t, $iid$ laplace and $iid$ lognormal. Every single one of these is going to be a sub-subclass since they will use `_IID` and that uses `Process`. 

We will start with making the process $iid$ Normal in 2D. In general they are defined using mean and variance where both are 2D vectors because the process is 2D. We want to add a feature that instead of `normal(mean=[0.0, 2.0], sigma=[1.0, 0.5])` if we simply write `normal(mean=0.0, sigma=1.0)` it will be able to recognize it as a 2D array by repeating the values as `normal(mean=[0.0, 0.0], sigma=[1.0, 1.0])`. That is where the first helper comes ito picture.

In [15]:
def _pair(v, name="parameter"):
    a = np.asarray(v, float)
    if a.ndim == 0:
        return np.repeat(a, 2)
    if a.shape == (2,):
        return a.copy()
    raise ValueError(f"{name} must be a scalar (used for both coordinates) or a length-2 sequence")

When entry is already 2-element list, `return a.copy()` is simply saying not to change anything about the entry and returning a copy of the entry (a copy instead of the original one to avoid re-writings). Everything else raises Error. 

In [16]:
class normal(_IID):
    """iid Normal(mean, sigma) noise at every time step (sigma = standard deviation)."""

    def __init__(self, mean=0.0, sigma=1.0, seed=None):
        m, s = _pair(mean, "mean"), _pair(sigma, "sigma")
        super().__init__([stats.norm(m[k], s[k]) for k in range(2)], seed,
                         f"normal(mean={fmt(m)}, sigma={fmt(s)})")


class cauchy(_IID):
    """iid Cauchy(loc, scale) noise (no mean, no variance, heavy tails)."""

    def __init__(self, loc=0.0, scale=1.0, seed=None):
        l, s = _pair(loc, "loc"), _pair(scale, "scale")
        super().__init__([stats.cauchy(l[k], s[k]) for k in range(2)], seed,
                         f"cauchy(loc={fmt(l)}, scale={fmt(s)})")

class student_t(_IID):
    """iid Student-t(df, loc, scale) noise."""

    def __init__(self, df=3.0, loc=0.0, scale=1.0, seed=None):
        d, l, s = _pair(df, "df"), _pair(loc, "loc"), _pair(scale, "scale")
        super().__init__([stats.t(d[k], l[k], s[k]) for k in range(2)], seed,
                         f"student_t(df={fmt(d)}, loc={fmt(l)}, scale={fmt(s)})")


class laplace(_IID):
    """iid Laplace(loc, scale) noise."""

    def __init__(self, loc=0.0, scale=1.0, seed=None):
        l, s = _pair(loc, "loc"), _pair(scale, "scale")
        super().__init__([stats.laplace(l[k], s[k]) for k in range(2)], seed,
                         f"laplace(loc={fmt(l)}, scale={fmt(s)})")


There is a little change that we do with `exp`, `unif` and `lognorm`.

We will do a change of variable. The user will give a rate, but scipy’s exponential takes a scale, and `scale = 1/rate`. So the class will convert. This is done so that the idea converges with the notations we see in books, we generally write "exp(rate)" or $exp(\lambda)$ not $exp(\frac{1}{\lambda})$.

Similarly, `scipy` neeeds `start` and `width` for `uniform`. That is we are supposed to write "unif(a, b-a)". In books, we see this as "unif(a,b)" and we are going to use that for this library, so we are going to make an internal conversion once user uploads $(a, b)$.

A lognormal variable is `exp(Normal(mean, sigma))`. The user gives the `mean` and `sigma` of the underlying normal, but scipy wants the shape `sigma` and a `scale` equal to `exp(mean)`. So the class converts the mean with `np.exp`. Note that mean here is not the mean of the lognormal itself, only of the normal inside the exponential.

All three of these parameter changes are done to fall back to the more natural parameter set ups that we are used to seeing in books.

In [17]:
class exponential(_IID):
    """iid Exponential(rate) noise."""

    def __init__(self, rate=1.0, seed=None):
        r = _pair(rate, "rate")
        super().__init__([stats.expon(scale=1 / r[k]) for k in range(2)], seed,
                         f"exponential(rate={fmt(r)})")

class uniform(_IID):
    """iid Uniform(low, high) noise."""

    def __init__(self, low=0.0, high=1.0, seed=None):
        a, b = _pair(low, "low"), _pair(high, "high")
        super().__init__([stats.uniform(a[k], b[k] - a[k]) for k in range(2)], seed,
                         f"uniform(low={fmt(a)}, high={fmt(b)})")

class lognormal(_IID):
    """iid LogNormal noise: exp(Normal(mean, sigma))."""

    def __init__(self, mean=0.0, sigma=1.0, seed=None):
        m, s = _pair(mean, "mean"), _pair(sigma, "sigma")
        super().__init__([stats.lognorm(s[k], scale=np.exp(m[k])) for k in range(2)], seed,
                         f"lognormal(mean={fmt(m)}, sigma={fmt(s)})")

Most importantly keep track of the fact that we have now the inherited powers such as `simulate`, `marginal` and `logpdf` (from `_IID`), which further helps us inherit `sample`, `sample_slices`, `pdf`, `center` (last two are yet to be defined but they are built on top of `marginal`). 

The $iid$ family is almost done. The only loose end that remains are `pdf` and `center`. We said earlier that once `_IID` has marginal, the base class’s `pdf()` and `center()` work automatically, and we haven’t shown you those yet. The $iid$ is the easiest case to understand them in, so we’ll do `pdf` now and `center` right after that.

We already saw how `sample` and `sample_slices` looks at the law of the process $(X_t, Y_t)$ through a cloud of sample points. A cloud is only an approximation and that cannot be used to draw the density surface next to the paths as we wanted. If the law is known exactly, we can compute its density; then we will have a number saying how likely it is for the process to land near a given point $(x, y)$. High value means likely, low value means unlikely. The viewer uses this to draw the smooth surface next to the paths. For the iid processes, X and Y are independent. For independent things, the joint density is just the product of the two separate densities and we already know the separate densities. That was what `marginal` is for since it returns the distribution of one coordinate.

In [20]:
def pdf(self, t, x, y, start=None):
    if not self.independent:
        return None
    s = self._start(start)
    out = 1.0
    for k, c in enumerate((x, y)):
        d = self.marginal(t, s, k)
        f = d.pmf(np.rint(c)) if isinstance(d.dist, stats.rv_discrete) else d.pdf(c)
        out = out * f
    return out

Explaination:

1. `if not self.independent: return None`. The product rule only works for independent coordinates. `independent` is a flag defined at the top of the class, `True` by default. A process with correlated coordinates, like Brownian motion with `rho` ≠ $0$, switches it to `False`. Then this method gives up and returns `None`, meaning “no simple answer here”. Such processes can supply their own pdf instead. We will see them in the next subsections where we go to Jump processes and Brownian process families.
2. `for k, c in enumerate((x, y))` loops twice. The first time `k=0, c=x`, the second time `k=1, c=y`. So `k` is the coordinate number and `c` is the point’s value in that coordinate.
3. `d = self.marginal(t, s, k)` gets the distribution of coordinate `k`. For `_IID` this is just the stored frozen distribution.
4. The `f = ...` line picks the right function, and it’s the only tricky line. We have already seen what `isinstance` does. Some distributions are continuous (Normal, Cauchy), and for those `d.pdf(c)` gives a density. Others are discrete, meaning they only take whole-number values (for example the Poisson process later in the file), and for those the right quantity is `d.pmf(...)`, the probability of that exact value. `isinstance(d.dist, stats.rv_discrete)` asks “is this a discrete distribution?”. The `np.rint(c)` rounds `c` to the nearest whole number first, so a point like $2.0000001$ doesn’t get a probability of zero.

5. Since it will loop twice and we start at `out = 1.0` we get the following: $$1.0 \longrightarrow 1.0* f(x) \longrightarrow \{1.0*f(x)\}*f(y)$$

Why this lives in the base class?

It is written once and uses only `marginal`. Any process that can say “here is the distribution of coordinate k at time t” gets pdf for free, as long as its coordinates are independent. For iid, that’s all the classes at once.

Lastly, `center` is where the cloud is centered. Basically the mean and we will not do the main hard work since that will be handled by `scipy` itself. We will just ask for the mean.

In [22]:
def center(self, t, start=None):
    s = self._start(start)
    if t <= 0:
        return s.copy()
    out = []
    for k in range(self.dim):
        d = self.marginal(t, s, k)
        m = d.mean()
        out.append(m if np.isfinite(m) else d.median())
    return np.array(out, float)

Explaination:
1. `s = self._start(start)` cleans the starting point, as always.
2. `if t <= 0: return s.copy()` handles time zero (or earlier). At time 0 nothing has happened yet, so the process is sitting exactly at its start point. `.copy()` hands back a fresh array so nobody can accidentally change `s` as we often do to avoid re-writings.
3. `for k in range(self.dim)` loops over the two coordinates, k = 0 for X and k = 1 for Y as always.
4. `d = self.marginal(t, s, k)` gets the distribution of that coordinate. It’s the same marginal we’ve now used three times.
5. As mentioned earlier, `scipy` will do the main job. `m = d.mean()` asks `scipy` for the average of that distribution.
6. `m if np.isfinite(m) else d.median()` is the interesting line. It means “use the mean, unless it isn’t a real finite number, in which case use the median.” This is specifically done keeping Cauchy in mind. This `center` is NOT just for calculating mean for the viewer, it also tells us where to aim the camera later on. So `nan` mean values returning the value "infinity" is not good enough, we need a point to focus.
7. `out.append()` simply appends the two center_x and center_y into `[center_x, center_y]`. That is why here we started with empty `out = []` whereas for `pdf` we had `out = 1.0` because there we were taking products in loop.

The full updated code as of now is the following:

In [23]:
from __future__ import annotations

import numpy as np
from scipy import stats

# ----------------------------------------------------------------------------------------- helpers
def _pair(v, name="parameter"):
    a = np.asarray(v, float)
    if a.ndim == 0:
        return np.repeat(a, 2)
    if a.shape == (2,):
        return a.copy()
    raise ValueError(f"{name} must be a scalar (used for both coordinates) or a length-2 sequence")

# --------------------------------------------------------------------------------------- base class
class Process:
    """Base class: every process knows how to simulate itself and (where possible) its law."""

    name = "process"
    dim = 2
    default_start = (0.0, 0.0)
    independent = True
    _desc = "process"

    def __init__(self, seed=None):
        self.seed = seed

    def __repr__(self):
        return self._desc

    def _start(self, start):
        """Validates or assigns default starting vector (0.0, 0.0)."""
        s = self.default_start if start is None else start
        s = np.asarray(s, float)
        if s.ndim == 0:
            s = np.repeat(s, self.dim)
        if s.shape != (self.dim,):
            raise ValueError(f"start must have {self.dim} coordinates")
        return s

    def _rng(self, seed=None):
        return np.random.default_rng(self.seed if seed is None else seed)

    #------------------------------------------------------------------ to be provided by subclasses
    def simulate(self, n, times, start, rng):
        """Paths of shape (n, len(times), dim); times[0] must be 0 and X_0 = start."""
        raise NotImplementedError

    def marginal(self, t, start, k):
        """Frozen scipy distribution of coordinate k at time t (when known in closed form)."""
        raise NotImplementedError

    #-----------------------------------------------------------------------------------------------

    def pdf(self, t, x, y, start=None):
        """Closed-form density (pmf for lattice laws) of (X_t, Y_t) at (x, y), or None.
        
        Product rule for independent coordinates: f(x, y) = f(x) * f(y)
        """
        if not self.independent:
            return None
        s = self._start(start)
        out = 1.0
        for k, c in enumerate((x, y)):
            d = self.marginal(t, s, k)
            f = d.pmf(np.rint(c)) if isinstance(d.dist, stats.rv_discrete) else d.pdf(c)
            out = out * f
        return out

    def center(self, t, start=None):
        """Mean of X_t (median if the mean does not exist, e.g. Cauchy)."""
        s = self._start(start)
        if t <= 0:
            return s.copy()
        out = []
        for k in range(self.dim):
            d = self.marginal(t, s, k)
            m = d.mean()
            out.append(m if np.isfinite(m) else d.median())
        return np.array(out, float)
    
    def sample_slices(self, times, n, rng, start, fine=None):
        """Simulates up to slice times and extracts the state at those times. Exact for all built-in processes."""
        X = self.simulate(n, np.r_[0.0, np.asarray(times, float)], start, rng)
        return np.transpose(X[:, 1:, :], (1, 0, 2))
        # Transpose from (n, slices, dim) -> (slices, n, dim)

    def paths(self, n=20, T=1.0, steps=200, start=None, seed=None):
        """Simulate n paths on [0, T]; returns (times, X) with X of shape (n, steps+1, 2)."""
        times = np.linspace(0.0, T, steps + 1)
        return times, self.simulate(n, times, self._start(start), self._rng(seed))

    def sample(self, t, n=10000, start=None, seed=None):
        """n independent samples of X_t, shape (n, 2) ie, at fixed time."""
        return self.sample_slices([t], n, self._rng(seed), self._start(start))[0]

# ------------------------------------------------------------------------------------ iid processes
class _IID(Process):
    """X_t independent across time; coordinates independent scipy distributions."""

    def __init__(self, frozen, seed=None, desc="iid"):
        super().__init__(seed)
        self.frozen = list(frozen)
        self._desc = desc

    def simulate(self, n, times, start, rng):
        out = np.empty((n, len(times), 2))
        out[:, 0, :] = start
        for k in range(2):
            out[:, 1:, k] = self.frozen[k].rvs(size=(n, len(times) - 1), random_state=rng)
        return out

    def marginal(self, t, start, k):
        return self.frozen[k]

    def logpdf(self, X):
        return sum(self.frozen[k].logpdf(X[..., k]) for k in range(2))


def iid(dist, seed=None):
    """iid(scipy.stats.gamma(2)) or iid([frozen_x, frozen_y]): any scipy distribution."""
    pair = list(dist) if isinstance(dist, (list, tuple)) else [dist, dist]
    return _IID(pair, seed, desc=f"iid({pair[0].dist.name})")


class normal(_IID):
    """iid Normal(mean, sigma) noise at every time step (sigma = standard deviation)."""

    def __init__(self, mean=0.0, sigma=1.0, seed=None):
        m, s = _pair(mean, "mean"), _pair(sigma, "sigma")
        super().__init__([stats.norm(m[k], s[k]) for k in range(2)], seed,
                         f"normal(mean={fmt(m)}, sigma={fmt(s)})")


class cauchy(_IID):
    """iid Cauchy(loc, scale) noise (no mean, no variance, heavy tails)."""

    def __init__(self, loc=0.0, scale=1.0, seed=None):
        l, s = _pair(loc, "loc"), _pair(scale, "scale")
        super().__init__([stats.cauchy(l[k], s[k]) for k in range(2)], seed,
                         f"cauchy(loc={fmt(l)}, scale={fmt(s)})")


class exponential(_IID):
    """iid Exponential(rate) noise."""

    def __init__(self, rate=1.0, seed=None):
        r = _pair(rate, "rate")
        super().__init__([stats.expon(scale=1 / r[k]) for k in range(2)], seed,
                         f"exponential(rate={fmt(r)})")


class uniform(_IID):
    """iid Uniform(low, high) noise."""

    def __init__(self, low=0.0, high=1.0, seed=None):
        a, b = _pair(low, "low"), _pair(high, "high")
        super().__init__([stats.uniform(a[k], b[k] - a[k]) for k in range(2)], seed,
                         f"uniform(low={fmt(a)}, high={fmt(b)})")


class student_t(_IID):
    """iid Student-t(df, loc, scale) noise."""

    def __init__(self, df=3.0, loc=0.0, scale=1.0, seed=None):
        d, l, s = _pair(df, "df"), _pair(loc, "loc"), _pair(scale, "scale")
        super().__init__([stats.t(d[k], l[k], s[k]) for k in range(2)], seed,
                         f"student_t(df={fmt(d)}, loc={fmt(l)}, scale={fmt(s)})")


class laplace(_IID):
    """iid Laplace(loc, scale) noise."""

    def __init__(self, loc=0.0, scale=1.0, seed=None):
        l, s = _pair(loc, "loc"), _pair(scale, "scale")
        super().__init__([stats.laplace(l[k], s[k]) for k in range(2)], seed,
                         f"laplace(loc={fmt(l)}, scale={fmt(s)})")


class lognormal(_IID):
    """iid LogNormal noise: exp(Normal(mean, sigma))."""

    def __init__(self, mean=0.0, sigma=1.0, seed=None):
        m, s = _pair(mean, "mean"), _pair(sigma, "sigma")
        super().__init__([stats.lognorm(s[k], scale=np.exp(m[k])) for k in range(2)], seed,
                         f"lognormal(mean={fmt(m)}, sigma={fmt(s)})")

## 2.1.2 Brownian family